# Explainable AutoML for Customer Churn Prediction

## Dissertation SHAP Analysis

This notebook explains the selected Azure AutoML XGBoost model used in the customer churn study. It produces reproducible global and local explanations suitable for the dissertation.

The analysis addresses four questions:

1. Which original customer variables have the greatest overall influence on churn predictions?
2. How do the main engineered features affect the direction and strength of predictions?
3. Why does the model classify one customer as churn and another as non-churn?
4. Do the SHAP explanations reproduce the probabilities produced by the underlying XGBoost model?

The notebook uses:

- Azure AutoML child run: `automl-riyadh-safe-v3-rai-01_1`
- Validation data asset: `customer-retention-validation-safe-v3:1`
- Target variable: `is_churn`
- A reproducible stratified sample of 3,000 validation records

The local explanations use probabilities from the same underlying XGBoost model explained by SHAP. This corrects the inconsistency found in the earlier notebook.

## 1. Install the notebook packages

Use the `Python 3.10 - SDK v2` kernel on `cpu-cluster`.

Run this cell once. When it finishes, restart the kernel and continue from Section 2.

In [ ]:
%pip install --quiet --upgrade-strategy only-if-needed \
    "azureml-core==1.61.0" \
    "azure-ai-ml" \
    "azure-identity" \
    "mltable" \
    "azureml-dataprep[pandas]" \
    "azure-mgmt-authorization<5" \
    "pandas<3" \
    "pyyaml"

print("Package installation finished.")
print("Restart the kernel once, then continue from Section 2.")

## 2. Configuration and output folders

Separate folders are used so the new dissertation analysis does not overwrite the earlier SHAP work.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import pandas as pd

EXPERIMENT_NAME = "customer-retention-riyadh-v3"
CHILD_RUN_ID = "automl-riyadh-safe-v3-rai-01_1"

DATA_ASSET_NAME = "customer-retention-validation-safe-v3"
DATA_ASSET_VERSION = "1"
TARGET_COLUMN = "is_churn"

SAMPLE_SIZE = 3000
RANDOM_STATE = 42

ROOT_DIR = Path.cwd()
BUNDLE_DIR = ROOT_DIR / "dissertation_shap_bundle"
OUTPUT_DIR = ROOT_DIR / "dissertation_shap_outputs"

MODEL_PATH = BUNDLE_DIR / "model.pkl"
ENVIRONMENT_YAML = BUNDLE_DIR / "automl_environment.yml"
CLEAN_ENVIRONMENT_YAML = BUNDLE_DIR / "automl_environment_clean.yml"
SAMPLE_PATH = BUNDLE_DIR / "validation_shap_sample.csv"
SCHEMA_PATH = BUNDLE_DIR / "validation_schema.json"
WORKER_SCRIPT = BUNDLE_DIR / "dissertation_shap_worker.py"

ENV_PREFIX = Path.home() / ".conda" / "envs" / "dissertation-shap-env"
ENV_PYTHON = ENV_PREFIX / "bin" / "python"

BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Working folder:", ROOT_DIR.resolve())
print("Bundle folder:", BUNDLE_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())

## 3. Download the selected AutoML model

The exact `model.pkl` and Conda environment are downloaded from the completed XGBoost child run. This keeps the explanation linked to the model evaluated in the dissertation.

In [ ]:
from azureml.core import Workspace, Experiment, Run

workspace = Workspace.from_config()
experiment = Experiment(
    workspace=workspace,
    name=EXPERIMENT_NAME,
)
child_run = Run(
    experiment=experiment,
    run_id=CHILD_RUN_ID,
)

run_status = child_run.get_details().get("status")

print("Workspace:", workspace.name)
print("Experiment:", EXPERIMENT_NAME)
print("Child run:", CHILD_RUN_ID)
print("Status:", run_status)

if run_status != "Completed":
    raise RuntimeError(
        f"The selected AutoML run is not complete. Status: {run_status}"
    )

available_files = child_run.get_file_names()

required_files = {
    "outputs/model.pkl": MODEL_PATH,
    "outputs/conda_env_v_1_0_0.yml": ENVIRONMENT_YAML,
}

for remote_path, local_path in required_files.items():
    if remote_path not in available_files:
        raise FileNotFoundError(
            f"Required AutoML file is missing: {remote_path}"
        )

    if local_path.exists():
        local_path.unlink()

    child_run.download_file(
        name=remote_path,
        output_file_path=str(local_path),
    )

    if not local_path.exists() or local_path.stat().st_size == 0:
        raise FileNotFoundError(
            f"The file was not downloaded correctly: {local_path}"
        )

    print(
        f"Downloaded {remote_path} "
        f"({local_path.stat().st_size:,} bytes)"
    )

print("MODEL DOWNLOAD COMPLETED")

## 4. Load the validation data

The validation asset is loaded through Azure ML. The target column and class balance are checked before sampling.

In [ ]:
import mltable
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
ml_client = MLClient.from_config(
    credential=credential
)

data_asset = ml_client.data.get(
    name=DATA_ASSET_NAME,
    version=DATA_ASSET_VERSION,
)

print("Data asset:", data_asset.name)
print("Version:", data_asset.version)
print("Type:", data_asset.type)

validation_table = None
load_errors = []

try:
    validation_table = mltable.load(
        f"azureml:/{data_asset.id}"
    )
    print("Loaded using the full Azure ML asset ID.")
except Exception as error:
    load_errors.append(
        ("full asset ID", repr(error))
    )

if validation_table is None:
    try:
        validation_table = mltable.load(
            f"azureml:{DATA_ASSET_NAME}:{DATA_ASSET_VERSION}",
            ml_client=ml_client,
        )
        print("Loaded using the short Azure ML asset URI.")
    except Exception as error:
        load_errors.append(
            ("short asset URI", repr(error))
        )

if validation_table is None:
    asset_path = getattr(
        data_asset,
        "path",
        None,
    )

    if asset_path:
        try:
            validation_table = mltable.load(
                str(asset_path)
            )
            print("Loaded using the underlying asset path.")
        except Exception as error:
            load_errors.append(
                ("underlying path", repr(error))
            )

if validation_table is None:
    for method, message in load_errors:
        print(f"{method}: {message}")

    raise RuntimeError(
        "The validation MLTable could not be loaded."
    )

validation_df = (
    validation_table
    .to_pandas_dataframe()
)

if TARGET_COLUMN not in validation_df.columns:
    raise KeyError(
        f"Target column '{TARGET_COLUMN}' was not found."
    )

validation_df[TARGET_COLUMN] = pd.to_numeric(
    validation_df[TARGET_COLUMN],
    errors="raise",
).astype(int)

print("\nValidation shape:", validation_df.shape)
print("\nTarget counts:")
print(
    validation_df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
)

print("\nTarget percentages:")
print(
    validation_df[TARGET_COLUMN]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

display(validation_df.head())

## 5. Create a reproducible stratified SHAP sample

A 3,000-row sample is used because it provides broad validation coverage while remaining practical for repeated SHAP calculations. Stratification preserves the original churn proportion.

In [ ]:
sample_size = min(
    SAMPLE_SIZE,
    len(validation_df),
)

class_counts = (
    validation_df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
)

class_ratios = (
    class_counts
    / class_counts.sum()
)

requested_counts = {
    int(label): int(
        round(
            sample_size
            * float(ratio)
        )
    )
    for label, ratio in class_ratios.items()
}

count_difference = (
    sample_size
    - sum(requested_counts.values())
)

if count_difference != 0:
    largest_class = int(
        class_counts.idxmax()
    )
    requested_counts[largest_class] += (
        count_difference
    )

sampled_groups = []

for label, count in sorted(
    requested_counts.items()
):
    class_frame = validation_df[
        validation_df[TARGET_COLUMN] == label
    ]

    sampled_groups.append(
        class_frame.sample(
            n=count,
            random_state=RANDOM_STATE + label,
            replace=False,
        )
    )

sample_df = (
    pd.concat(
        sampled_groups,
        ignore_index=True,
    )
    .sample(
        frac=1,
        random_state=RANDOM_STATE,
    )
    .reset_index(drop=True)
)

sample_df.to_csv(
    SAMPLE_PATH,
    index=False,
)

schema = {
    column: str(dtype)
    for column, dtype
    in validation_df.dtypes.items()
}

SCHEMA_PATH.write_text(
    json.dumps(
        schema,
        indent=2,
    ),
    encoding="utf-8",
)

print("Sample saved:", SAMPLE_PATH.resolve())
print("Schema saved:", SCHEMA_PATH.resolve())
print("Sample shape:", sample_df.shape)

print("\nSample target counts:")
print(
    sample_df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
)

print("\nSample target percentages:")
print(
    sample_df[TARGET_COLUMN]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

## 6. Create the compatible model environment

The AutoML model is loaded in its own Conda environment rather than in the notebook kernel. This avoids scikit-learn compatibility errors.

This cell also:

- accepts the required Anaconda channel terms;
- installs SHAP and Matplotlib;
- pins Setuptools below version 81 so `pkg_resources` remains available for Azure AutoML dependencies;
- verifies that the saved model can be loaded.

Set `RECREATE_ENV = True` only when a complete rebuild is required.

In [ ]:
conda_candidates = [
    Path("/anaconda/bin/conda"),
    Path("/anaconda/condabin/conda"),
    Path("/anaconda/envs/azureml_py310_sdkv2/bin/conda"),
    Path("/anaconda/envs/azureml_py38/bin/conda"),
]

conda_on_path = shutil.which("conda")

if conda_on_path:
    conda_candidates.append(
        Path(conda_on_path)
    )

CONDA_EXE = next(
    (
        path
        for path in conda_candidates
        if path.exists()
    ),
    None,
)

if CONDA_EXE is None:
    raise FileNotFoundError(
        "A Conda executable could not be found."
    )

print("Conda executable:", CONDA_EXE)

channels = [
    "https://repo.anaconda.com/pkgs/main",
    "https://repo.anaconda.com/pkgs/r",
]

for channel in channels:
    result = subprocess.run(
        [
            str(CONDA_EXE),
            "tos",
            "accept",
            "--override-channels",
            "--channel",
            channel,
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            f"Terms could not be accepted for {channel}"
        )

cleaned_lines = []

for line in ENVIRONMENT_YAML.read_text(
    encoding="utf-8"
).splitlines():
    if (
        line.startswith("name:")
        or line.startswith("prefix:")
    ):
        continue

    cleaned_lines.append(line)

CLEAN_ENVIRONMENT_YAML.write_text(
    "\n".join(cleaned_lines) + "\n",
    encoding="utf-8",
)

RECREATE_ENV = False

if RECREATE_ENV and ENV_PREFIX.exists():
    subprocess.run(
        [
            str(CONDA_EXE),
            "env",
            "remove",
            "--prefix",
            str(ENV_PREFIX),
            "--yes",
        ],
        check=False,
    )

    if ENV_PREFIX.exists():
        shutil.rmtree(
            ENV_PREFIX,
            ignore_errors=True,
        )

if not ENV_PYTHON.exists():
    print(
        "Creating the AutoML environment. "
        "This may take several minutes."
    )

    create_result = subprocess.run(
        [
            str(CONDA_EXE),
            "env",
            "create",
            "--prefix",
            str(ENV_PREFIX),
            "--file",
            str(CLEAN_ENVIRONMENT_YAML),
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(create_result.stdout)

    if create_result.returncode != 0:
        raise RuntimeError(
            "Conda environment creation failed."
        )
else:
    print(
        "Reusing the existing environment:",
        ENV_PREFIX,
    )

if not ENV_PYTHON.exists():
    raise FileNotFoundError(
        f"Environment Python was not created: {ENV_PYTHON}"
    )

repair_result = subprocess.run(
    [
        str(ENV_PYTHON),
        "-m",
        "pip",
        "install",
        "--force-reinstall",
        "--no-cache-dir",
        "--no-deps",
        "setuptools==80.9.0",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(repair_result.stdout)

if repair_result.returncode != 0:
    raise RuntimeError(
        "The Setuptools compatibility repair failed."
    )

package_result = subprocess.run(
    [
        str(ENV_PYTHON),
        "-m",
        "pip",
        "install",
        "--upgrade-strategy",
        "only-if-needed",
        "shap==0.44.0",
        "matplotlib==3.7.5",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(package_result.stdout)

if package_result.returncode != 0:
    raise RuntimeError(
        "SHAP or Matplotlib installation failed."
    )

verification_code = """
import sys
import joblib
import pkg_resources
import sklearn
import shap
import xgboost

model = joblib.load(sys.argv[1])

print("Python:", sys.executable)
print("scikit-learn:", sklearn.__version__)
print("SHAP:", shap.__version__)
print("XGBoost:", xgboost.__version__)
print("pkg_resources:", pkg_resources.__file__)
print("Model type:", type(model))
print("MODEL LOAD TEST PASSED")
"""

verification_result = subprocess.run(
    [
        str(ENV_PYTHON),
        "-c",
        verification_code,
        str(MODEL_PATH),
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(verification_result.stdout)

if verification_result.returncode != 0:
    raise RuntimeError(
        "The compatible environment could not load model.pkl."
    )

print("ENVIRONMENT PREPARATION COMPLETED")

## 7. SHAP analysis design

To improve dissertation readability, engineered SHAP values are grouped back to the original 29 input variables for the main global importance chart and the two local waterfall plots. The detailed beeswarm is retained at engineered-feature level as technical evidence.

The local churn and non-churn cases are selected from correctly classified customers using probabilities from the same underlying XGBoost model explained by SHAP. The code reconstructs each probability from the SHAP base value and feature contributions. The analysis stops if the reconstructed probability differs from the XGBoost probability by more than 0.01.

In [ ]:
%%writefile dissertation_shap_bundle/dissertation_shap_worker.py

from __future__ import annotations

import argparse
import json
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from scipy import sparse
from scipy.special import expit
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def find_pipeline(model_object):
    """Return the fitted sklearn-style pipeline stored in the AutoML model."""

    queue = [model_object]
    checked = set()
    possible_attributes = [
        "pipeline",
        "_pipeline",
        "fitted_pipeline",
        "_fitted_pipeline",
        "model",
        "_model",
    ]

    while queue:
        current = queue.pop(0)

        if current is None or id(current) in checked:
            continue

        checked.add(id(current))

        if hasattr(current, "steps"):
            return current

        for attribute in possible_attributes:
            try:
                candidate = getattr(current, attribute, None)
            except Exception:
                candidate = None

            if candidate is not None:
                queue.append(candidate)

        if isinstance(current, dict):
            queue.extend(current.values())
        elif isinstance(current, (list, tuple)):
            queue.extend(current)

    raise TypeError(
        f"A fitted pipeline could not be found inside {type(model_object)}."
    )


def restore_dtypes(frame, schema):
    """Restore the basic Pandas dtypes recorded before CSV export."""

    restored = frame.copy()

    for column, dtype_name in schema.items():
        if column not in restored.columns:
            continue

        dtype_name = dtype_name.lower()

        if dtype_name.startswith("float"):
            restored[column] = pd.to_numeric(
                restored[column],
                errors="coerce",
            ).astype(float)

        elif dtype_name.startswith("int"):
            restored[column] = pd.to_numeric(
                restored[column],
                errors="raise",
            ).astype(int)

        elif dtype_name in {"bool", "boolean"}:
            restored[column] = restored[column].astype(bool)

        else:
            restored[column] = restored[column].astype(object)

    return restored


def get_engineered_feature_names(pipeline, feature_count):
    """Read AutoML engineered feature names where the transformer exposes them."""

    for _, step in pipeline.steps:
        method = getattr(step, "get_engineered_feature_names", None)

        if method is None:
            continue

        try:
            names = [str(name) for name in method()]

            if len(names) == feature_count:
                return names

        except Exception as error:
            print("Feature-name warning:", error)

    return [
        f"engineered_feature_{index}"
        for index in range(feature_count)
    ]


def normalise_shap_values(values, feature_count):
    """Convert different binary-class SHAP return formats to one 2-D array."""

    if isinstance(values, list):
        array = np.asarray(values[-1])
    else:
        array = np.asarray(values)

    if array.ndim == 3:
        array = array[:, :, -1]

    if array.ndim != 2:
        raise ValueError(
            f"Unexpected SHAP dimensions: {array.shape}"
        )

    if array.shape[1] != feature_count:
        raise ValueError(
            f"SHAP returned {array.shape[1]} features, "
            f"but the engineered data has {feature_count}."
        )

    return array


def map_engineered_to_raw(engineered_name, raw_columns):
    """Map an AutoML engineered feature back to its original input variable."""

    direct_matches = [
        column
        for column in raw_columns
        if (
            engineered_name == column
            or engineered_name.startswith(column + "_")
            or engineered_name.startswith(column + ".")
        )
    ]

    if direct_matches:
        return max(direct_matches, key=len)

    contained_matches = [
        column
        for column in raw_columns
        if column in engineered_name
    ]

    if contained_matches:
        return max(contained_matches, key=len)

    return engineered_name


def group_shap_by_raw_feature(
    shap_values,
    engineered_names,
    raw_columns,
):
    """Sum engineered SHAP contributions into the original input variables."""

    mapping_rows = []
    grouped_columns = []

    for raw_feature in raw_columns:
        indices = [
            index
            for index, engineered_name in enumerate(engineered_names)
            if map_engineered_to_raw(
                engineered_name,
                raw_columns,
            ) == raw_feature
        ]

        if indices:
            grouped_columns.append(
                shap_values[:, indices].sum(axis=1)
            )
        else:
            grouped_columns.append(
                np.zeros(shap_values.shape[0])
            )

    grouped_values = np.column_stack(grouped_columns)

    for engineered_name in engineered_names:
        mapping_rows.append(
            {
                "engineered_feature": engineered_name,
                "raw_feature": map_engineered_to_raw(
                    engineered_name,
                    raw_columns,
                ),
            }
        )

    mapping_frame = pd.DataFrame(mapping_rows)

    return grouped_values, mapping_frame


def select_representative_case(
    candidate_indices,
    probabilities,
    target_probability,
):
    """Select a correctly classified case close to a readable probability."""

    if len(candidate_indices) == 0:
        raise ValueError(
            "No correctly classified case was available for the local explanation."
        )

    distances = np.abs(
        probabilities[candidate_indices]
        - target_probability
    )

    return int(
        candidate_indices[np.argmin(distances)]
    )


def save_grouped_importance_chart(importance_frame, output_path):
    """Save a dissertation-friendly bar chart using original feature names."""

    top_features = (
        importance_frame
        .head(15)
        .sort_values(
            "mean_absolute_shap",
            ascending=True,
        )
    )

    plt.figure(figsize=(10, 7))
    plt.barh(
        top_features["raw_feature"],
        top_features["mean_absolute_shap"],
    )
    plt.xlabel("Mean absolute SHAP value")
    plt.ylabel("Original input feature")
    plt.title(
        "Global SHAP Importance Grouped by Original Feature"
    )
    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close("all")


def save_engineered_beeswarm(
    shap_values,
    engineered_features,
    output_path,
):
    """Save the detailed engineered-feature beeswarm plot."""

    plt.figure()

    shap.summary_plot(
        shap_values,
        engineered_features,
        max_display=20,
        show=False,
    )

    plt.title(
        "Detailed SHAP Distribution for Engineered Features"
    )
    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close("all")


def choose_dependence_feature(
    engineered_names,
    engineered_importance,
):
    """Choose a meaningful continuous feature for the dependence plot."""

    preferred_features = [
        "days_since_last_transaction_MeanImputer",
        "transaction_history_days_MeanImputer",
        "days_since_last_activity_MeanImputer",
        "plan_list_price_mean_MeanImputer",
        "actual_amount_paid_mean_MeanImputer",
    ]

    for feature_name in preferred_features:
        if feature_name in engineered_names:
            return engineered_names.index(feature_name)

    ranked_indices = np.argsort(
        engineered_importance
    )[::-1]

    for index in ranked_indices:
        feature_name = engineered_names[int(index)]

        if "CharGramCountVectorizer" not in feature_name:
            return int(index)

    return int(ranked_indices[0])


def save_dependence_plot(
    feature_index,
    shap_values,
    engineered_features,
    output_path,
):
    """Save a dependence plot for a meaningful numerical feature."""

    feature_name = engineered_features.columns[
        feature_index
    ]

    plt.figure()
    shap.dependence_plot(
        feature_index,
        shap_values,
        engineered_features,
        interaction_index="auto",
        show=False,
    )
    plt.title(
        f"SHAP Dependence: {feature_name}"
    )
    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close("all")

    return feature_name


def save_grouped_waterfall(
    grouped_values,
    base_value,
    raw_columns,
    title,
    output_path,
):
    """Save a local waterfall using original feature names."""

    explanation = shap.Explanation(
        values=grouped_values,
        base_values=base_value,
        feature_names=raw_columns,
    )

    shap.plots.waterfall(
        explanation,
        max_display=15,
        show=False,
    )

    plt.title(title)
    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close("all")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--sample", required=True)
    parser.add_argument("--schema", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--target", required=True)
    args = parser.parse_args()

    model_path = Path(args.model).resolve()
    sample_path = Path(args.sample).resolve()
    schema_path = Path(args.schema).resolve()
    output_dir = Path(args.output).resolve()

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    sample_frame = pd.read_csv(sample_path)
    schema = json.loads(
        schema_path.read_text(
            encoding="utf-8"
        )
    )
    sample_frame = restore_dtypes(
        sample_frame,
        schema,
    )

    if args.target not in sample_frame.columns:
        raise KeyError(
            f"Target column '{args.target}' was not found."
        )

    X_raw = sample_frame.drop(
        columns=[args.target]
    ).copy()

    y_true = (
        pd.to_numeric(
            sample_frame[args.target],
            errors="raise",
        )
        .astype(int)
        .to_numpy()
    )

    raw_columns = list(X_raw.columns)

    print("Raw predictors:", X_raw.shape)
    print("Target:", y_true.shape)

    saved_model = joblib.load(model_path)
    pipeline = find_pipeline(saved_model)

    print("Loaded model type:", type(saved_model))
    print("Pipeline steps:")

    for number, (name, step) in enumerate(
        pipeline.steps,
        start=1,
    ):
        print(
            f"{number}. {name}: "
            f"{type(step).__module__}.{type(step).__name__}"
        )

    engineered_data = X_raw.copy()

    for step_name, transformer in pipeline.steps[:-1]:
        if transformer is None:
            continue

        if (
            isinstance(transformer, str)
            and transformer == "passthrough"
        ):
            continue

        engineered_data = transformer.transform(
            engineered_data
        )

        print(
            f"After {step_name}: "
            f"{engineered_data.shape}"
        )

    if sparse.issparse(engineered_data):
        engineered_array = engineered_data.toarray()
    else:
        engineered_array = np.asarray(
            engineered_data
        )

    engineered_array = engineered_array.astype(float)

    engineered_names = get_engineered_feature_names(
        pipeline,
        engineered_array.shape[1],
    )

    engineered_frame = pd.DataFrame(
        engineered_array,
        columns=engineered_names,
    )

    _, wrapped_classifier = pipeline.steps[-1]

    if hasattr(wrapped_classifier, "get_model"):
        tree_model = wrapped_classifier.get_model()
    else:
        tree_model = wrapped_classifier

    if tree_model is None:
        raise RuntimeError(
            "The underlying XGBoost model could not be extracted."
        )

    print("Underlying model:", type(tree_model))

    probability_matrix = np.asarray(
        tree_model.predict_proba(
            engineered_array
        )
    )

    classes = np.asarray(
        getattr(
            tree_model,
            "classes_",
            [0, 1],
        )
    )

    positive_columns = np.where(
        classes == 1
    )[0]

    if len(positive_columns) != 1:
        raise ValueError(
            f"Class 1 could not be identified from {classes}."
        )

    positive_column = int(
        positive_columns[0]
    )

    churn_probability = probability_matrix[
        :,
        positive_column,
    ]

    predicted_class = (
        churn_probability >= 0.5
    ).astype(int)

    explainer = shap.TreeExplainer(
        tree_model
    )

    try:
        raw_shap_values = explainer.shap_values(
            engineered_frame,
            check_additivity=False,
        )
    except TypeError:
        raw_shap_values = explainer.shap_values(
            engineered_frame
        )

    shap_values = normalise_shap_values(
        raw_shap_values,
        engineered_frame.shape[1],
    )

    expected_values = np.asarray(
        explainer.expected_value
    ).reshape(-1)

    base_value = float(
        expected_values[-1]
    )

    shap_margin = (
        base_value
        + shap_values.sum(axis=1)
    )
    shap_probability = expit(
        shap_margin
    )

    direct_error = np.mean(
        np.abs(
            shap_probability
            - churn_probability
        )
    )

    reverse_error = np.mean(
        np.abs(
            (1 - shap_probability)
            - churn_probability
        )
    )

    if reverse_error < direct_error:
        shap_values = -shap_values
        base_value = -base_value
        shap_margin = (
            base_value
            + shap_values.sum(axis=1)
        )
        shap_probability = expit(
            shap_margin
        )

    probability_difference = np.abs(
        shap_probability
        - churn_probability
    )

    maximum_probability_difference = float(
        probability_difference.max()
    )

    if maximum_probability_difference > 0.01:
        raise RuntimeError(
            "SHAP reconstruction did not agree with "
            "the underlying XGBoost probabilities."
        )

    grouped_shap_values, mapping_frame = (
        group_shap_by_raw_feature(
            shap_values,
            engineered_names,
            raw_columns,
        )
    )

    raw_importance = pd.DataFrame(
        {
            "raw_feature": raw_columns,
            "mean_absolute_shap": np.abs(
                grouped_shap_values
            ).mean(axis=0),
        }
    ).sort_values(
        "mean_absolute_shap",
        ascending=False,
    ).reset_index(drop=True)

    engineered_importance_values = np.abs(
        shap_values
    ).mean(axis=0)

    engineered_importance = pd.DataFrame(
        {
            "engineered_feature": engineered_names,
            "mean_absolute_shap": (
                engineered_importance_values
            ),
        }
    ).sort_values(
        "mean_absolute_shap",
        ascending=False,
    ).reset_index(drop=True)

    raw_importance.to_csv(
        output_dir
        / "01_grouped_raw_shap_importance.csv",
        index=False,
    )

    engineered_importance.to_csv(
        output_dir
        / "02_engineered_shap_importance.csv",
        index=False,
    )

    mapping_frame.to_csv(
        output_dir
        / "03_engineered_to_raw_mapping.csv",
        index=False,
    )

    save_grouped_importance_chart(
        raw_importance,
        output_dir
        / "04_grouped_raw_shap_bar.png",
    )

    save_engineered_beeswarm(
        shap_values,
        engineered_frame,
        output_dir
        / "05_engineered_shap_beeswarm.png",
    )

    dependence_index = choose_dependence_feature(
        engineered_names,
        engineered_importance_values,
    )

    dependence_feature = save_dependence_plot(
        dependence_index,
        shap_values,
        engineered_frame,
        output_dir
        / "06_shap_dependence.png",
    )

    correct_churn = np.where(
        (y_true == 1)
        & (predicted_class == 1)
    )[0]

    correct_nonchurn = np.where(
        (y_true == 0)
        & (predicted_class == 0)
    )[0]

    churn_index = select_representative_case(
        correct_churn,
        churn_probability,
        target_probability=0.85,
    )

    nonchurn_index = select_representative_case(
        correct_nonchurn,
        churn_probability,
        target_probability=0.15,
    )

    churn_title = (
        "Local SHAP Explanation: Correctly Predicted Churn "
        f"(p={churn_probability[churn_index]:.3f})"
    )

    nonchurn_title = (
        "Local SHAP Explanation: Correctly Predicted Non-Churn "
        f"(p={churn_probability[nonchurn_index]:.3f})"
    )

    save_grouped_waterfall(
        grouped_shap_values[churn_index],
        base_value,
        raw_columns,
        churn_title,
        output_dir
        / "07_waterfall_churn_grouped.png",
    )

    save_grouped_waterfall(
        grouped_shap_values[nonchurn_index],
        base_value,
        raw_columns,
        nonchurn_title,
        output_dir
        / "08_waterfall_nonchurn_grouped.png",
    )

    prediction_evidence = pd.DataFrame(
        {
            "actual_class": y_true,
            "predicted_class": predicted_class,
            "xgboost_churn_probability": (
                churn_probability
            ),
            "shap_reconstructed_probability": (
                shap_probability
            ),
            "absolute_probability_difference": (
                probability_difference
            ),
        }
    )

    prediction_evidence.to_csv(
        output_dir
        / "09_prediction_evidence.csv",
        index=False,
    )

    metrics = {
        "sample_records": int(
            len(y_true)
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                predicted_class,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predicted_class,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                predicted_class,
                zero_division=0,
            )
        ),
        "f1_score": float(
            f1_score(
                y_true,
                predicted_class,
                zero_division=0,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                churn_probability,
            )
        ),
    }

    pd.DataFrame(
        [metrics]
    ).to_csv(
        output_dir
        / "10_sample_performance_metrics.csv",
        index=False,
    )

    matrix = confusion_matrix(
        y_true,
        predicted_class,
        labels=[0, 1],
    )

    confusion_frame = pd.DataFrame(
        matrix,
        index=[
            "actual_non_churn",
            "actual_churn",
        ],
        columns=[
            "predicted_non_churn",
            "predicted_churn",
        ],
    )

    confusion_frame.to_csv(
        output_dir
        / "11_confusion_matrix.csv"
    )

    selected_cases = []

    for label, index in [
        ("churn", churn_index),
        ("non_churn", nonchurn_index),
    ]:
        selected_record = X_raw.iloc[
            [index]
        ].copy()

        selected_record[
            "actual_class"
        ] = int(
            y_true[index]
        )
        selected_record[
            "predicted_class"
        ] = int(
            predicted_class[index]
        )
        selected_record[
            "xgboost_churn_probability"
        ] = float(
            churn_probability[index]
        )
        selected_record[
            "shap_reconstructed_probability"
        ] = float(
            shap_probability[index]
        )

        selected_record.to_csv(
            output_dir
            / (
                "12_selected_churn_record.csv"
                if label == "churn"
                else "13_selected_nonchurn_record.csv"
            ),
            index=False,
        )

        selected_cases.append(
            {
                "case": label,
                "row_index": int(index),
                "actual_class": int(
                    y_true[index]
                ),
                "predicted_class": int(
                    predicted_class[index]
                ),
                "xgboost_churn_probability": float(
                    churn_probability[index]
                ),
                "shap_reconstructed_probability": float(
                    shap_probability[index]
                ),
                "raw_margin": float(
                    shap_margin[index]
                ),
            }
        )

    local_consistency = pd.DataFrame(
        selected_cases
    )

    local_consistency.to_csv(
        output_dir
        / "14_local_shap_consistency.csv",
        index=False,
    )

    local_contributions = []

    for case_name, index in [
        ("churn", churn_index),
        ("non_churn", nonchurn_index),
    ]:
        for raw_feature, contribution in zip(
            raw_columns,
            grouped_shap_values[index],
        ):
            local_contributions.append(
                {
                    "case": case_name,
                    "raw_feature": raw_feature,
                    "shap_contribution": float(
                        contribution
                    ),
                }
            )

    pd.DataFrame(
        local_contributions
    ).to_csv(
        output_dir
        / "15_local_shap_contributions.csv",
        index=False,
    )

    metadata = {
        "analysis_scope": (
            "SHAP analysis on a stratified 3,000-row "
            "validation sample"
        ),
        "raw_feature_count": int(
            X_raw.shape[1]
        ),
        "engineered_feature_count": int(
            engineered_frame.shape[1]
        ),
        "shap_base_value_raw_margin": (
            base_value
        ),
        "maximum_probability_difference": (
            maximum_probability_difference
        ),
        "dependence_feature": (
            dependence_feature
        ),
        "churn_case_index": (
            churn_index
        ),
        "nonchurn_case_index": (
            nonchurn_index
        ),
        "important_note": (
            "Performance metrics in this output describe "
            "the stratified SHAP sample, not the full validation set."
        ),
    }

    (
        output_dir
        / "16_run_metadata.json"
    ).write_text(
        json.dumps(
            metadata,
            indent=2,
        ),
        encoding="utf-8",
    )

    zip_path = (
        output_dir.parent
        / "dissertation_shap_outputs.zip"
    )

    if zip_path.exists():
        zip_path.unlink()

    shutil.make_archive(
        str(
            zip_path.with_suffix("")
        ),
        "zip",
        root_dir=str(output_dir),
    )

    print("\nTop grouped raw features:")
    print(
        raw_importance.head(15).to_string(
            index=False
        )
    )

    print("\nSample performance metrics:")
    print(
        pd.DataFrame(
            [metrics]
        ).to_string(
            index=False
        )
    )

    print("\nLocal explanation consistency:")
    print(
        local_consistency.to_string(
            index=False
        )
    )

    print(
        "\nMaximum model-versus-SHAP probability difference:",
        maximum_probability_difference,
    )

    print("\nSHAP analysis completed successfully.")
    print("Results folder:", output_dir)
    print("ZIP archive:", zip_path)


if __name__ == "__main__":
    main()


## 8. Run the SHAP analysis

This cell clears only the new dissertation output folder and runs the worker program.

In [ ]:
if OUTPUT_DIR.exists():
    for item in OUTPUT_DIR.iterdir():
        if item.is_file():
            item.unlink()
        elif item.is_dir():
            shutil.rmtree(item)
else:
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

command = [
    str(ENV_PYTHON),
    str(WORKER_SCRIPT),
    "--model",
    str(MODEL_PATH),
    "--sample",
    str(SAMPLE_PATH),
    "--schema",
    str(SCHEMA_PATH),
    "--output",
    str(OUTPUT_DIR),
    "--target",
    TARGET_COLUMN,
]

print("Running dissertation SHAP analysis...\n")

run_result = subprocess.run(
    command,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(run_result.stdout)

if run_result.returncode != 0:
    raise RuntimeError(
        "The SHAP analysis failed. "
        "Read the final traceback above."
    )

print(
    "\nDISSERTATION SHAP ANALYSIS "
    "COMPLETED SUCCESSFULLY"
)

## 9. Review the results

The main dissertation evidence is:

- grouped global feature importance using original variable names;
- detailed engineered-feature beeswarm;
- dependence plot for a meaningful numerical variable;
- grouped local explanation for a correctly predicted churn customer;
- grouped local explanation for a correctly predicted non-churn customer;
- probability consistency evidence;
- sample-level performance metrics and confusion matrix.

The performance table describes the 3,000-row SHAP sample. It should not replace the full validation metrics reported for the AutoML run.

In [ ]:
from IPython.display import Image, display

required_outputs = [
    "01_grouped_raw_shap_importance.csv",
    "02_engineered_shap_importance.csv",
    "03_engineered_to_raw_mapping.csv",
    "04_grouped_raw_shap_bar.png",
    "05_engineered_shap_beeswarm.png",
    "06_shap_dependence.png",
    "07_waterfall_churn_grouped.png",
    "08_waterfall_nonchurn_grouped.png",
    "09_prediction_evidence.csv",
    "10_sample_performance_metrics.csv",
    "11_confusion_matrix.csv",
    "12_selected_churn_record.csv",
    "13_selected_nonchurn_record.csv",
    "14_local_shap_consistency.csv",
    "15_local_shap_contributions.csv",
    "16_run_metadata.json",
]

missing_outputs = [
    file_name
    for file_name in required_outputs
    if not (OUTPUT_DIR / file_name).exists()
]

if missing_outputs:
    raise FileNotFoundError(
        "Missing outputs: "
        + ", ".join(missing_outputs)
    )

raw_importance = pd.read_csv(
    OUTPUT_DIR
    / "01_grouped_raw_shap_importance.csv"
)

sample_metrics = pd.read_csv(
    OUTPUT_DIR
    / "10_sample_performance_metrics.csv"
)

local_consistency = pd.read_csv(
    OUTPUT_DIR
    / "14_local_shap_consistency.csv"
)

print("Top grouped raw features:")
display(raw_importance.head(15))

print("Sample-level performance:")
display(sample_metrics)

print("Local probability consistency:")
display(local_consistency)

for image_name in [
    "04_grouped_raw_shap_bar.png",
    "05_engineered_shap_beeswarm.png",
    "06_shap_dependence.png",
    "07_waterfall_churn_grouped.png",
    "08_waterfall_nonchurn_grouped.png",
]:
    print("\n", image_name)

    display(
        Image(
            filename=str(
                OUTPUT_DIR / image_name
            )
        )
    )

zip_path = (
    ROOT_DIR
    / "dissertation_shap_outputs.zip"
)

print("\nResults folder:")
print(OUTPUT_DIR.resolve())

print("\nZIP archive:")
print(zip_path.resolve())

print("\nNOTEBOOK COMPLETED SUCCESSFULLY")

## 10. Dissertation reporting guidance

Use the grouped raw-feature bar chart as the main global explanation because it presents the results in terms of recognisable customer variables. The engineered beeswarm can be used in the detailed results section or appendix to demonstrate how Azure AutoML transformed categorical variables.

Interpret the dependence plot as an association within the fitted model rather than evidence of causation. The local waterfall plots explain two selected predictions and should not be treated as representative of every customer.

When reporting the findings, distinguish clearly between:

- global importance across the SHAP sample;
- local contributions for individual customers;
- full AutoML validation metrics;
- sample-level metrics generated for explainability checking.

This distinction supports a critical Level 7 discussion of transparency, practical usefulness and methodological limitations.